# CE49X Lab 3: Where Should You Open a Gas Station in Istanbul?
## A Traffic-Based Site Selection Analysis

**Instructor:** Dr. Eyuphan Koc  
**Department of Civil Engineering, Bogazici University**  
**Semester:** Spring 2026

---

## Background

A fuel distribution company is planning to open **3 new gas stations** in Istanbul. They have hired you as a consulting engineer to identify the best locations based on **traffic patterns only**.

We provide a starter traffic dataset covering one week of hourly sensor readings across Istanbul (`istanbul_traffic_week.csv` + `sensor_coords.csv`). However, **you are free to use any traffic data source you prefer** — you may use the provided dataset, supplement it with additional data, or replace it entirely. Some options:

- **Provided dataset:** `istanbul_traffic_week.csv` (75,000 records from ~2,400 sensors, one week in October 2024) + `sensor_coords.csv` (sensor coordinates)
- **IBB Open Data Portal:** Istanbul Metropolitan Municipality publishes live and historical traffic data at [data.ibb.gov.tr](https://data.ibb.gov.tr). You can query their APIs for broader coverage or more recent data.
- **Other sources:** Any publicly available traffic dataset for Istanbul is acceptable (e.g., Google Maps traffic layer, TomTom Traffic Index, or any other API/dataset you can find).

**Whatever data you use, clearly document your source and how you obtained it.**

Your job is to:
1. **Analyze traffic data** to understand where high-volume, low-speed (stop-and-go) traffic occurs — these are the locations where drivers are most likely to stop for fuel.
2. **Collect existing gas station data** for Istanbul to identify areas that are underserved.
3. **Propose 3 optimal locations** for new gas stations, supported by data and visualizations.

## Provided Data (Optional Starting Point)

The following files are included in the course repository. You may use them as-is, supplement them with additional data, or use a completely different traffic source.

### `istanbul_traffic_week.csv`

| Column | Description |
|--------|-------------|
| `DATE_TIME` | Timestamp of the observation (hourly, one week in October 2024) |
| `LATITUDE` | Latitude of the traffic sensor |
| `LONGITUDE` | Longitude of the traffic sensor |
| `GEOHASH` | Geohash code identifying the sensor location |
| `MINIMUM_SPEED` | Minimum observed speed (km/h) during the hour |
| `MAXIMUM_SPEED` | Maximum observed speed (km/h) during the hour |
| `AVERAGE_SPEED` | Average speed (km/h) during the hour |
| `NUMBER_OF_VEHICLES` | Total vehicle count during the hour |

### `sensor_coords.csv`

| Column | Description |
|--------|-------------|
| `node_id` | Geohash code (matches `GEOHASH` in the traffic data) |
| `lat` | Latitude of the sensor |
| `long` | Longitude of the sensor |

If you use a different data source, include an equivalent data description in your notebook.

## Deliverables

Your notebook must include the following:

### 1. Traffic Data — Source & Exploration
- **Document your traffic data source.** If you use the provided dataset, state that. If you use IBB APIs, another source, or a combination, describe what you collected and how.
- Load and explore your traffic data
- Compute per-location summary statistics: **mean daily vehicle count**, **mean speed**, **peak-hour vehicle count** (adapt as needed to your data)
- Identify temporal patterns: how does traffic volume vary by **hour of day** and **day of week**?
- Identify the **top 20 highest-traffic locations** by total vehicle count

### 2. Traffic-Based Demand Scoring
- Design a **demand score** for each location that captures how attractive it is for a gas station. Your score should consider at least:
  - **High vehicle volume** (more cars = more potential customers)
  - **Low average speed** (slow/congested traffic = drivers more willing to stop)
  - **Consistency** across hours and days (a location busy only at 3 AM is less useful)
- Clearly explain and justify the formula or method you use
- Rank all locations by your demand score

### 3. Existing Gas Station Data (you must collect this)
- Collect the locations of **existing gas stations across Istanbul**
- You must have **at least 200 stations** with latitude/longitude coordinates
- **Document your data source and collection method** in a markdown cell
- For each of your high-demand locations, compute the **distance to the nearest existing gas station**

### 4. Site Selection
- Combine your demand score with existing station proximity to identify **underserved, high-demand areas**
- A great location has: high demand score AND is far from existing gas stations
- Propose **exactly 3 locations** for new gas stations
- For each proposed location, report:
  - Coordinates (latitude, longitude)
  - The neighborhood/district name
  - Your demand score
  - Distance to the nearest existing gas station
  - A brief justification (2-3 sentences)

### 5. Visualizations
- Create **at least three plots/maps**. Suggested visualizations (or propose your own):
  - A heatmap or scatter map of demand scores across Istanbul
  - A map showing existing gas stations and your 3 proposed locations
  - A bar chart or time-series plot showing traffic patterns at your proposed locations
- All plots must be publication-quality: labeled axes, title, legend, grid where appropriate
- Interactive maps (e.g., folium) are encouraged but not required

### 6. Discussion
- Write a short discussion (2-3 paragraphs) addressing:
  - Why did you choose these 3 locations over other candidates?
  - What **limitations** does a traffic-only analysis have? What other factors would a real site selection study consider (e.g., land cost, zoning, competition, road type)?
  - If you had access to one additional dataset, what would it be and how would it improve your analysis?

## Hints

- **Haversine formula** for distance between two GPS coordinates:

$$d = 2R \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1)\cos(\phi_2)\sin^2\left(\frac{\Delta\lambda}{2}\right)}\right)$$

  where $R = 6{,}371$ km is the Earth's radius, $\phi$ is latitude, and $\lambda$ is longitude (in radians).

- **Normalizing scores:** When combining metrics with different scales (e.g., vehicle count vs. speed), normalize each to a 0-1 range first:

$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

- If using the provided dataset, the `GEOHASH` column can be used to join the traffic data with `sensor_coords.csv` via the `node_id` column.

- Think about whether **weekday** vs. **weekend** traffic patterns matter for a gas station business.

## Grading

| Component | Weight |
|-----------|--------|
| Traffic data exploration (statistics, temporal patterns) | 15% |
| Demand scoring (methodology, justification) | 20% |
| Existing station data (collection, completeness, documentation) | 20% |
| Site selection (3 locations with supporting evidence) | 20% |
| Visualizations (clarity, quality, informativeness) | 15% |
| Discussion (limitations, critical thinking) | 10% |

## Submission

1. Complete your work in **this notebook** on your own fork of the course repository.
2. Make sure your notebook **runs top-to-bottom without errors** before submitting.
3. Commit and push your completed notebook to your fork.
4. We will grade directly from your fork — there is no separate upload. Make sure your latest work is pushed before the deadline.

---
## Your Work Starts Here

In [ ]:
# ============================================================
# Istanbul Gas Station Site Selection Analysis
# CE49X - Week 03 Assignment
# Data: istanbul_traffic_week.csv + sensor_coords.csv + OSM gas stations
# ============================================================

from pathlib import Path
import warnings

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

plt.rcParams.update(
    {
        "figure.facecolor": "#0f1117",
        "axes.facecolor": "#1a1d2e",
        "axes.edgecolor": "#333",
        "text.color": "white",
        "axes.labelcolor": "#aaa",
        "xtick.color": "#aaa",
        "ytick.color": "#aaa",
        "grid.color": "#333",
        "grid.alpha": 0.4,
    }
)


def resolve_data_file(filename: str) -> Path:
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend(
            [
                base / filename,
                base / "lab" / filename,
                base / "Week03_NumPy_Pandas" / filename,
                base / "Week03_NumPy_Pandas" / "lab" / filename,
            ]
        )

    seen = set()
    for path in candidates:
        if path in seen:
            continue
        seen.add(path)
        if path.exists():
            return path

    raise FileNotFoundError(f"Could not find {filename}")


traffic_path = resolve_data_file("istanbul_traffic_week.csv")
coords_path = resolve_data_file("sensor_coords.csv")
gas_path = resolve_data_file("istanbul_gas_stations.csv")

# ============================================================
# SECTION 1 - LOAD DATA
# ============================================================

df = pd.read_csv(traffic_path)
df["DATE_TIME"] = pd.to_datetime(df["DATE_TIME"])
df["HOUR"] = df["DATE_TIME"].dt.hour
df["DOW"] = df["DATE_TIME"].dt.dayofweek
df["DATE"] = df["DATE_TIME"].dt.date
df["IS_WEEKEND"] = df["DOW"].isin([5, 6])

print(f"Traffic records  : {len(df):,}")
print(f"Unique sensors   : {df['GEOHASH'].nunique():,}")
print(f"Date range       : {df['DATE_TIME'].min()} to {df['DATE_TIME'].max()}")
print(df.head())

coords = pd.read_csv(coords_path).rename(
    columns={"node_id": "GEOHASH", "lat": "LAT", "long": "LON"}
)
coords = coords[["GEOHASH", "LAT", "LON"]]
print(f"\nSensor coords: {len(coords):,} entries")

gs = pd.read_csv(gas_path)[["latitude", "longitude"]].rename(
    columns={"latitude": "lat", "longitude": "lon"}
)
gs = gs[(gs["lat"].between(40.75, 41.40)) & (gs["lon"].between(27.9, 30.0))]
gs = gs.dropna().reset_index(drop=True)
print(f"Gas stations     : {len(gs):,}")

# ============================================================
# SECTION 2 - PER-LOCATION SUMMARY STATISTICS
# ============================================================

num_days = max(df["DATE"].nunique(), 1)
loc = (
    df.groupby("GEOHASH")
    .agg(
        mean_daily_vehicles=("NUMBER_OF_VEHICLES", lambda x: x.sum() / num_days),
        mean_speed=("AVERAGE_SPEED", "mean"),
        peak_vehicles=("NUMBER_OF_VEHICLES", "max"),
        total_vehicles=("NUMBER_OF_VEHICLES", "sum"),
        obs_count=("NUMBER_OF_VEHICLES", "count"),
    )
    .reset_index()
)

traffic_coords = (
    df.groupby("GEOHASH")
    .agg(LAT=("LATITUDE", "median"), LON=("LONGITUDE", "median"))
    .reset_index()
)
coords = coords.merge(traffic_coords, on="GEOHASH", how="outer", suffixes=("", "_traffic"))
coords["LAT"] = coords["LAT"].fillna(coords["LAT_traffic"])
coords["LON"] = coords["LON"].fillna(coords["LON_traffic"])
coords = coords[["GEOHASH", "LAT", "LON"]].dropna()

loc = loc.merge(coords, on="GEOHASH", how="left").dropna(subset=["LAT", "LON"])

print(f"\nLocations with data: {len(loc):,}")
print("\nTop 20 highest-traffic locations:")
print(
    loc.nlargest(20, "total_vehicles")
    [["GEOHASH", "LAT", "LON", "total_vehicles", "mean_speed", "peak_vehicles"]]
    .to_string(index=False)
)

# ============================================================
# SECTION 3 - TEMPORAL PATTERNS
# ============================================================

hourly = df.groupby("HOUR")["NUMBER_OF_VEHICLES"].mean()
daily = df.groupby("DOW")["NUMBER_OF_VEHICLES"].mean()
wd_avg = df[~df["IS_WEEKEND"]].groupby("HOUR")["NUMBER_OF_VEHICLES"].mean()
we_avg = df[df["IS_WEEKEND"]].groupby("HOUR")["NUMBER_OF_VEHICLES"].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
bar_colors = [
    "#ff6b6b" if h in list(range(7, 10)) + list(range(17, 20)) else "#4fc3f7"
    for h in hourly.index
]
ax.bar(hourly.index, hourly.values, color=bar_colors, alpha=0.85, width=0.8)
ax.plot(wd_avg.index, wd_avg.values, color="#ffd700", lw=2, label="Weekday avg", zorder=3)
ax.plot(we_avg.index, we_avg.values, color="#ff9800", lw=2, ls="--", label="Weekend avg", zorder=3)
ax.axvspan(6.5, 9.5, alpha=0.12, color="yellow")
ax.axvspan(16.5, 19.5, alpha=0.12, color="orange")
ax.set_title("Avg Vehicles by Hour of Day", fontsize=13, fontweight="bold")
ax.set_xlabel("Hour")
ax.set_ylabel("Avg Number of Vehicles")
ax.set_xticks(range(0, 24, 2))
ax.legend()
ax.grid(True, axis="y")

ax = axes[1]
day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
day_vals = [daily.get(i, 0) for i in range(7)]
wd_colors = ["#4fc3f7"] * 5 + ["#ff6b6b"] * 2
ax.bar(day_labels, day_vals, color=wd_colors, alpha=0.85, width=0.6)
ax.set_title("Avg Vehicles by Day of Week", fontsize=13, fontweight="bold")
ax.set_xlabel("Day")
ax.set_ylabel("Avg Number of Vehicles")
ax.grid(True, axis="y")
ax.legend(
    handles=[
        mpatches.Patch(color="#4fc3f7", label="Weekday"),
        mpatches.Patch(color="#ff6b6b", label="Weekend"),
    ]
)

plt.suptitle("Istanbul Traffic Temporal Patterns - October 2024", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("plot_1_temporal.png", dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("\nPlot 1 saved: plot_1_temporal.png")

# ============================================================
# SECTION 4 - DEMAND SCORE
# ============================================================


def normalize(series: pd.Series) -> pd.Series:
    mn, mx = series.min(), series.max()
    return (series - mn) / (mx - mn + 1e-9)


loc["v_norm"] = normalize(loc["total_vehicles"])
loc["s_norm"] = normalize(loc["mean_speed"])
loc["p_norm"] = normalize(loc["peak_vehicles"])
loc["demand_score"] = (
    0.5 * loc["v_norm"] + 0.3 * (1 - loc["s_norm"]) + 0.2 * loc["p_norm"]
)

print("\nDemand score distribution:")
print(loc["demand_score"].describe().round(3))

fig, ax = plt.subplots(figsize=(12, 9))
sc = ax.scatter(
    loc["LON"],
    loc["LAT"],
    c=loc["demand_score"],
    cmap="YlOrRd",
    s=loc["demand_score"] * 80 + 5,
    alpha=0.75,
    zorder=2,
)
plt.colorbar(sc, ax=ax, label="Demand Score", shrink=0.8)
ax.set_title("Traffic Demand Score Heatmap - Istanbul", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True)
plt.tight_layout()
plt.savefig("plot_2_demand_heatmap.png", dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("Plot 2 saved: plot_2_demand_heatmap.png")

# ============================================================
# SECTION 5 - GAS STATION GAP ANALYSIS (HAVERSINE)
# ============================================================


def haversine_min(lat: float, lon: float, gs_df: pd.DataFrame) -> float:
    """Return distance in km to the nearest gas station."""
    r = 6371.0
    lat1 = np.radians(lat)
    lon1 = np.radians(lon)
    lat2 = np.radians(gs_df["lat"].to_numpy())
    lon2 = np.radians(gs_df["lon"].to_numpy())
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return float((2 * r * np.arcsin(np.sqrt(a))).min())


loc["nearest_gs_km"] = loc.apply(lambda row: haversine_min(row["LAT"], row["LON"], gs), axis=1)

print("\nDistance to nearest gas station (km):")
print(loc["nearest_gs_km"].describe().round(3))

# ============================================================
# SECTION 6 - COMBINED SCORE & SITE SELECTION
# ============================================================

loc["gap_norm"] = normalize(loc["nearest_gs_km"])
loc["final_score"] = 0.65 * loc["demand_score"] + 0.35 * loc["gap_norm"]

candidates = loc.sort_values("final_score", ascending=False).reset_index(drop=True)

print("\nTop 20 candidate sites (combined score):")
print(
    candidates[["GEOHASH", "LAT", "LON", "demand_score", "nearest_gs_km", "final_score"]]
    .head(20)
    .to_string(index=False)
)


def haversine(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    r = 6371.0
    la1, lo1, la2, lo2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return float(2 * r * np.arcsin(np.sqrt(a)))


selected = []
for _, row in candidates.iterrows():
    if len(selected) == 3:
        break
    too_close = any(
        haversine(row["LAT"], row["LON"], chosen["LAT"], chosen["LON"]) < 2.0
        for chosen in selected
    )
    if not too_close:
        selected.append(row)

selected_df = pd.DataFrame(selected)

print("\n" + "=" * 55)
print("    3 PROPOSED GAS STATION LOCATIONS")
print("=" * 55)
for i, site in enumerate(selected, start=1):
    print(f"\nLocation {i}")
    print(f"   Geohash            : {site['GEOHASH']}")
    print(f"   Coordinates        : ({site['LAT']:.5f}, {site['LON']:.5f})")
    print(f"   Demand Score       : {site['demand_score']:.3f}")
    print(f"   Nearest Station    : {site['nearest_gs_km']:.2f} km")
    print(f"   Final Score        : {site['final_score']:.3f}")

# ============================================================
# SECTION 7 - VISUALIZATIONS
# ============================================================

fig, ax = plt.subplots(figsize=(13, 10))
ax.scatter(gs["lon"], gs["lat"], c="#4fc3f7", s=15, alpha=0.45, label="Existing Gas Stations", zorder=2)
scatter = ax.scatter(loc["LON"], loc["LAT"], c=loc["demand_score"], cmap="Greens", s=25, alpha=0.35, zorder=1)
plt.colorbar(scatter, ax=ax, label="Demand Score", shrink=0.75)

colors_p = ["#ff4d4d", "#ffd700", "#00e676"]
for i, site in enumerate(selected, start=1):
    color = colors_p[i - 1]
    ax.scatter(
        site["LON"],
        site["LAT"],
        c=color,
        s=250,
        marker="*",
        zorder=5,
        edgecolors="white",
        linewidths=0.8,
        label=f"Proposed #{i}",
    )
    ax.annotate(
        f"#{i}",
        (site["LON"], site["LAT"]),
        textcoords="offset points",
        xytext=(7, 7),
        color=color,
        fontsize=10,
        fontweight="bold",
    )

ax.set_title("Existing Gas Stations vs. Proposed Locations - Istanbul", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(loc="upper left", facecolor="#1a1d2e", edgecolor="#555", labelcolor="white", fontsize=8)
ax.grid(True)
plt.tight_layout()
plt.savefig("plot_3_station_map.png", dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("Plot 3 saved: plot_3_station_map.png")

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(loc["nearest_gs_km"], loc["demand_score"], c=loc["final_score"], cmap="plasma", s=25, alpha=0.6, zorder=1)
for i, site in enumerate(selected, start=1):
    color = colors_p[i - 1]
    ax.scatter(
        site["nearest_gs_km"],
        site["demand_score"],
        c=color,
        s=250,
        marker="*",
        zorder=5,
        edgecolors="white",
        linewidths=0.8,
        label=f"Proposed #{i}",
    )
    ax.annotate(
        f"#{i}",
        (site["nearest_gs_km"], site["demand_score"]),
        textcoords="offset points",
        xytext=(6, 5),
        color=color,
        fontsize=10,
        fontweight="bold",
    )
ax.axvline(x=2.0, color="#ff6b6b", ls="--", alpha=0.6, label="2 km threshold")
ax.set_title("Demand Score vs. Distance to Nearest Gas Station", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Distance to Nearest Existing Gas Station (km)")
ax.set_ylabel("Demand Score")
ax.legend(facecolor="#1a1d2e", edgecolor="#555", labelcolor="white", fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.savefig("plot_4_demand_vs_gap.png", dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("Plot 4 saved: plot_4_demand_vs_gap.png")

fig, ax = plt.subplots(figsize=(12, 8))
top20 = loc.nlargest(20, "total_vehicles")
selected_hashes = set(selected_df["GEOHASH"]) if not selected_df.empty else set()
bar_colors = ["#ff4d4d" if geohash in selected_hashes else "#4fc3f7" for geohash in top20["GEOHASH"]]
ax.barh(range(len(top20)), top20["total_vehicles"].values, color=bar_colors, alpha=0.85)
ax.set_yticks(range(len(top20)))
ax.set_yticklabels(
    [f"{row.GEOHASH} ({row.LAT:.2f}, {row.LON:.2f})" for row in top20.itertuples()],
    fontsize=8,
    color="#ccc",
)
ax.invert_yaxis()
ax.set_title("Top 20 Sensors by Total Vehicle Count", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Total Vehicles (1 week)")
ax.legend(
    handles=[
        mpatches.Patch(color="#ff4d4d", label="Proposed Location"),
        mpatches.Patch(color="#4fc3f7", label="Other High-Traffic"),
    ],
    facecolor="#1a1d2e",
    edgecolor="#555",
    labelcolor="white",
)
ax.grid(True, axis="x")
plt.tight_layout()
plt.savefig("plot_5_top20_sensors.png", dpi=130, bbox_inches="tight", facecolor="#0f1117")
plt.show()
print("Plot 5 saved: plot_5_top20_sensors.png")

print("\nAnalysis complete. All plots saved.")


---

### Questions?

**Dr. Eyuphan Koc**  
eyuphan.koc@bogazici.edu.tr